# SurvFace Grad-CAM — 01. Origin embeddings and top-1 gallery templates

전체 선택 표본의 원본 512D 임베딩을 만들고 공식 registered/unmated probe에는 frozen origin top-1 gallery target을 고정합니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "survface"

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")

from research.experiments import extract_step4_origin_embeddings
from research.runtime import ProgressReporter

PROGRESS = ProgressReporter(
    "SurvFace Step 4 origin embeddings",
    heartbeat_seconds=None,
    milestone_percent=10,
)


In [2]:
if EXECUTE_STAGE:
    result = extract_step4_origin_embeddings(
        CONFIG_PATH,
        project_root=PROJECT_ROOT,
        dataset_id=DATASET_ID,
        execution_acknowledged=True,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:step4:"),
    )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


[22:18:10] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=2m 26s | progress=10% processed=46336 total=463341 rate=317.22/s eta=21m 55s
[22:20:30] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=4m 46s | progress=20% processed=92672 total=463341 rate=323.89/s eta=19m 04s
[22:22:40] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=6m 56s | progress=30% processed=139008 total=463341 rate=333.92/s eta=16m 11s
[22:24:51] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=9m 06s | progress=40% processed=185344 total=463341 rate=339.26/s eta=13m 39s
[22:27:10] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=11m 26s | progress=50% processed=231680 total=463341 rate=337.69/s eta=11m 26s
[22:29:28] SurvFace Step 4 origin embeddings | origin embedding extraction | elapsed=13m 43s | progress=60% processed=278016 total=463341 rate=337.66/s eta=9m 09s
[22:31:45] SurvFace Ste

{'run_id': '20260729-R001-cccb59d5',
 'dataset_id': 'survface',
 'samples': 463341,
 'saliency_target_eligible': 182159,
 'target_name': 'origin_top1_gallery_cosine',
 'next_stage': '02_population_gradcam_extraction'}

## 다음 단계

다음은 `experiment/00_population_gradcam_extraction.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.